# Colab으로 배우는 FastAPI: 스마트카 원격제어 API

**목표:** 스마트폰·브라우저가 보낸 명령을 FastAPI 서버가 검사하고 자동차 동작으로 연결하는 과정을 실습합니다.

> 오늘은 실제 모터 대신 `CAR_STATE`라는 가상 자동차를 제어합니다. 마지막에 같은 API를 Jetson Nano의 모터 함수와 연결하는 방법을 확인합니다.

**진행 순서**  
1. 서버와 API 개념 → 2. 첫 GET API → 3. 자동차 POST API → 4. 오류 실험 → 5. 휴대전화 접속(선택) → 6. 확장 미션

## 0. 시작 전 약속

- 셀은 위에서 아래로 한 번씩 실행합니다.
- 2인 1조라면 코더와 테스터 역할을 정하고 중간에 교대합니다.
- 임시 공개 URL과 `CLASS_KEY`는 수업 팀 밖에 공유하지 않습니다.
- 실제 차량 연결은 오늘 실습이 끝난 뒤, 교사 감독 아래 바퀴를 띄운 상태에서 진행합니다.

## 1. 핵심 개념 6개

| 개념 | 스마트카에서의 역할 |
|---|---|
| 클라이언트 | 명령을 보내는 브라우저·앱 |
| 서버 | 명령을 받아 처리하는 Jetson Nano/FastAPI |
| 엔드포인트 | 기능별 주소: `/health`, `/car/drive` |
| GET | 상태를 조회하는 요청 |
| POST | 주행 명령을 전달하는 요청 |
| JSON | 프로그램끼리 주고받는 구조화된 데이터 |

**생각하기:** `{"command":"forward","speed":40}`에서 키(key)와 값(value)은 무엇인가요?

## 2. 필요한 패키지 설치
Colab 런타임이 새로 시작될 때마다 이 셀을 먼저 실행합니다.

In [ ]:
!pip -q install fastapi uvicorn requests

## 3. FastAPI 앱 만들기

아래 코드는 다음을 만듭니다.

- `GET /health`: 서버 준비 상태
- `GET /car/state`: 현재 가상 자동차 상태
- `POST /car/drive`: 주행 명령
- `POST /car/stop`: 비상정지

`Field`의 범위 규칙 덕분에 속도는 0~100, 동작 시간은 100~3000ms만 허용됩니다.

In [ ]:
import secrets
from enum import Enum

from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field


app = FastAPI(
    title="Smart Car Classroom API",
    description="Colab에서 실행하는 가상 스마트카 원격제어 API",
    version="1.0.0",
)

CLASS_KEY = secrets.token_urlsafe(8)
CAR_STATE = {
    "command": "stop",
    "speed": 0,
    "last_duration_ms": 0,
}


class Direction(str, Enum):
    forward = "forward"
    backward = "backward"
    left = "left"
    right = "right"
    stop = "stop"


class DriveCommand(BaseModel):
    command: Direction
    speed: int = Field(default=40, ge=0, le=100)
    duration_ms: int = Field(default=500, ge=100, le=3000)


def check_class_key(x_class_key: str = Header(...)):
    if x_class_key != CLASS_KEY:
        raise HTTPException(status_code=401, detail="수업 키가 올바르지 않습니다.")


@app.get("/")
def root():
    return {
        "message": "Smart Car API is running",
        "docs": "/docs",
        "simulation": True,
    }


@app.get("/health")
def health():
    return {"status": "ok", "message": "server is ready"}


@app.get("/car/state")
def car_state():
    return CAR_STATE


@app.post("/car/drive")
def drive(command: DriveCommand, _: None = Depends(check_class_key)):
    speed = 0 if command.command == Direction.stop else command.speed
    CAR_STATE.update(
        command=command.command.value,
        speed=speed,
        last_duration_ms=command.duration_ms,
    )
    return {
        "accepted": True,
        "state": CAR_STATE,
        "note": "simulation only - no real motor is connected",
    }


@app.post("/car/stop")
def emergency_stop(_: None = Depends(check_class_key)):
    CAR_STATE.update(command="stop", speed=0, last_duration_ms=0)
    return {"accepted": True, "state": CAR_STATE, "message": "emergency stop"}


print("API 정의 완료")
print("오늘 조의 CLASS_KEY:", CLASS_KEY)

## 4. 서버 시작

Colab 안에서 Uvicorn 서버를 백그라운드 스레드로 시작합니다. 같은 셀을 다시 실행해도 이전 서버를 먼저 종료하도록 구성했습니다.

In [ ]:
import threading
import time
import uvicorn


try:
    if "server" in globals() and server is not None:
        server.should_exit = True
    if "server_thread" in globals() and server_thread.is_alive():
        server_thread.join(timeout=3)
except Exception:
    pass

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)

BASE_URL = "http://127.0.0.1:8000"
print("서버 실행 여부:", server_thread.is_alive())
print("Colab 내부 주소:", BASE_URL)

## 5. GET 요청: 서버 상태 확인
`status_code`가 200이고 JSON에 `status: ok`가 보이면 성공입니다.

In [ ]:
import requests

response = requests.get(f"{BASE_URL}/health", timeout=5)
print("상태코드:", response.status_code)
print("JSON 응답:", response.json())

### 체크 1

- 요청을 보낸 프로그램(클라이언트)은 무엇인가요?
- 응답을 만든 서버는 무엇인가요?
- 200은 어떤 뜻인가요?

## 6. POST 요청: 자동차 전진
주행 명령은 상태를 바꾸므로 POST로 전달합니다. 인증용 `x-class-key` 헤더도 함께 보냅니다.

In [ ]:
headers = {"x-class-key": CLASS_KEY}
payload = {
    "command": "forward",
    "speed": 40,
    "duration_ms": 500,
}

response = requests.post(
    f"{BASE_URL}/car/drive",
    json=payload,
    headers=headers,
    timeout=5,
)
print("상태코드:", response.status_code)
print("JSON 응답:", response.json())

## 7. 방향 바꾸기
아래 `command`를 `left`, `right`, `backward`, `stop`으로 바꾸어 각각 실행해 보세요.

In [ ]:
my_command = {
    "command": "left",       # left/right/backward/stop으로 바꾸기
    "speed": 30,
    "duration_ms": 700,
}

response = requests.post(
    f"{BASE_URL}/car/drive",
    json=my_command,
    headers=headers,
    timeout=5,
)
print(response.status_code, response.json())

## 8. 현재 상태 조회
마지막 명령이 자동차 상태에 남아 있는지 확인합니다.

In [ ]:
response = requests.get(f"{BASE_URL}/car/state", timeout=5)
print(response.status_code, response.json())

## 9. 오류도 중요한 응답이다

다음 두 오류를 일부러 만들어 봅니다.

- `speed=150` → 허용 범위 초과이므로 **422**
- 틀린 수업 키 → 인증 실패이므로 **401**

In [ ]:
bad_payload = {"command": "forward", "speed": 150, "duration_ms": 500}
response = requests.post(
    f"{BASE_URL}/car/drive",
    json=bad_payload,
    headers=headers,
    timeout=5,
)
print("상태코드:", response.status_code)
print("오류 내용:", response.json())

In [ ]:
wrong_headers = {"x-class-key": "wrong-key"}
response = requests.post(
    f"{BASE_URL}/car/stop",
    headers=wrong_headers,
    timeout=5,
)
print("상태코드:", response.status_code)
print("오류 내용:", response.json())

## 10. 비상정지
실차에서는 어떤 오류가 발생해도 최종 기본값이 정지가 되도록 설계해야 합니다.

In [ ]:
response = requests.post(
    f"{BASE_URL}/car/stop",
    headers=headers,
    timeout=5,
)
print(response.status_code, response.json())

## 11. 선택 실습: 휴대전화에서 `/docs` 열기

Cloudflare Quick Tunnel은 **개발·실습 전용 임시 공개 주소**를 만듭니다. 계정은 필요 없지만 학교 네트워크 정책에 따라 막힐 수 있습니다.

**주의**

- 생성된 URL은 인터넷에 공개됩니다.
- URL과 `CLASS_KEY`를 수업 팀 밖에 공유하지 마세요.
- 실제 모터가 연결된 Jetson을 이 방식으로 바로 공개하지 마세요.
- 끝나면 반드시 아래 종료 셀을 실행하세요.

In [ ]:
import os
import re
import subprocess
import time
import urllib.request


CLOUDFLARED = "/tmp/cloudflared"
if not os.path.exists(CLOUDFLARED):
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

try:
    if "tunnel_process" in globals() and tunnel_process.poll() is None:
        tunnel_process.terminate()
        tunnel_process.wait(timeout=5)
except Exception:
    pass

tunnel_process = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", BASE_URL, "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    if line:
        match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    elif tunnel_process.poll() is not None:
        break

if public_url:
    print("공개 API 주소:", public_url)
    print("Swagger 문서:", public_url + "/docs")
    print("x-class-key:", CLASS_KEY)
else:
    print("공개 주소를 만들지 못했습니다. 내부 requests 실습만 계속하세요.")

### 휴대전화 `/docs` 사용법

1. 출력된 `...trycloudflare.com/docs`를 휴대전화에서 엽니다.
2. `POST /car/drive` → **Try it out**을 누릅니다.
3. `x-class-key`에 출력된 수업 키를 입력합니다.
4. Request body의 command, speed, duration_ms를 바꿉니다.
5. **Execute** 후 상태코드와 응답 JSON을 확인합니다.

## 12. 확장 미션 — 하나를 선택하세요

1. `GET /car/battery`를 추가해 0~100 사이 값을 반환하기
2. `POST /car/honk`를 추가해 `honked: true` 반환하기
3. speed 기본값을 40에서 30으로 바꾸고 생략 테스트하기
4. 실제 차량의 watchdog 규칙을 글로 설계하기

아래 셀에 코드를 작성한 뒤 **API 앱 셀부터 서버 시작 셀까지 다시 실행**하세요.

In [ ]:
# 여기에 확장 기능을 작성하세요.
# 예시 시작:
# @app.get("/car/battery")
# def battery():
#     return {"battery_percent": 87}

<details>
<summary><b>확장 미션 예시 답안 보기</b></summary>

```python
@app.get("/car/battery")
def battery():
    return {"battery_percent": 87}

@app.post("/car/honk")
def honk(_: None = Depends(check_class_key)):
    return {"honked": True, "message": "빵빵! (simulation)"}
```
</details>

## 13. Jetson Nano 연결 구조

Colab에서는 `CAR_STATE`만 바꾸었습니다. Jetson에서는 엔드포인트와 JSON 형식은 유지하고, 아래 함수 안을 실제 모터 드라이버 호출로 교체합니다.

```python
def apply_motor(command, speed):
    if command == "forward":
        motor_driver.forward(speed)
    elif command == "left":
        motor_driver.left(speed)
    elif command == "right":
        motor_driver.right(speed)
    elif command == "backward":
        motor_driver.backward(speed)
    else:
        motor_driver.stop()
```

**실차 필수 안전장치:** 물리 비상정지, 속도·시간 상한, 통신 끊김 자동정지(watchdog), 인증, 로그, 바퀴를 띄운 첫 시험.

## 14. Exit ticket

1. 오늘 만든 GET 엔드포인트와 POST 엔드포인트를 하나씩 적으세요.
2. `speed=150`이 거절되는 이유와 상태코드를 적으세요.
3. 실제 차량 연결 전에 넣을 안전장치 2개를 적으세요.
4. Colab 실습 코드에서 Jetson 모터 함수로 교체할 부분은 어디인가요?

## 15. 실습 종료
공개 터널과 FastAPI 서버를 모두 종료합니다.

In [ ]:
try:
    if "tunnel_process" in globals() and tunnel_process.poll() is None:
        tunnel_process.terminate()
        tunnel_process.wait(timeout=5)
        print("공개 터널 종료 완료")
except Exception as e:
    print("터널 종료 확인:", e)

try:
    if "server" in globals() and server is not None:
        server.should_exit = True
    if "server_thread" in globals() and server_thread.is_alive():
        server_thread.join(timeout=5)
    print("FastAPI 서버 종료 완료")
except Exception as e:
    print("서버 종료 확인:", e)